# EODHD stock prices for the 110-name universe (Colab)

Pulls EODHD daily prices for all 206 names in the preserved 110-name
point-in-time universe (#281), so the first run can train on EODHD instead
of the LSEG export. The universe itself is unchanged: the same membership
file is byte-copied into the new package.

Each name's EODHD symbol is proven against the preserved LSEG panel: its
daily returns must agree over the span the universe needs it. A name that
does not agree stops the pull with its name, and no manifest is published.

**Before you start**

1. The preserved LSEG package is on Drive at
   `MCI_GRU_shared/preservation/2026-10-02-110-universe-2016`.
2. A Colab secret named `EODHD_API_KEY` exists and this notebook has access
   to it (key icon in the left bar). The key is never printed or written.
3. Any runtime type works; no GPU is needed.

**Where outputs go.** The package is built on local disk, because manifest
publication needs hard links. It is then copied to
`MyDrive/MCI_GRU_shared/preservation/<OUTPUT_NAME>/`, beside the LSEG
package. EODHD responses are cached on Drive, so a re-run after a symbol fix
spends calls only on what changed. Nothing on Drive is ever overwritten or
deleted.

In [ ]:
# Parameters
BRANCH = "main"
EXPECTED_COMMIT = ""  # optional: the full commit SHA to run

LSEG_PACKAGE_DRIVE_DIR = "/content/drive/MyDrive/MCI_GRU_shared/preservation/2026-10-02-110-universe-2016"
PREFIX = "sp500_pit_gics_top10_mcap_monthly_20160104_20260731"
REFERENCE_PANEL = f"market/{PREFIX}_lseg_20150101_20260731.csv"
PIT_UNIVERSE = f"constituents/{PREFIX}_pit_universe.csv"
REFERENCE_MANIFEST = f"data/manifests/{PREFIX}.r1.json"
REFERENCE_MANIFEST_SHA256 = "1e2043dcb4f00a88de128985cc94423c122e480034c16ec741710a2460653ecf"

START, END = "2015-01-01", "2026-07-31"  # the LSEG panel's span
OUTPUT_NAME = f"eodhd_{PREFIX}_r1"
OUTPUT_DRIVE_DIR = f"/content/drive/MyDrive/MCI_GRU_shared/preservation/{OUTPUT_NAME}"
CACHE_DRIVE_DIR = "/content/drive/MyDrive/MCI_GRU_shared/eodhd_cache/gics_top10_110_2016"
LOCAL_PACKAGE = "/content/eodhd_package"
LOCAL_MANIFEST = f"/content/{PREFIX}_eodhd.r1.json"

In [ ]:
# Mount Drive and read the EODHD key from Colab secrets. It is placed in this
# process's environment for the pull's subprocess only; never printed or saved.
import os
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
os.environ["EODHD_API_KEY"] = userdata.get("EODHD_API_KEY") or ""
if not os.environ["EODHD_API_KEY"].strip():
    raise RuntimeError("The Colab secret EODHD_API_KEY is empty or not shared with this notebook.")
print("EODHD_API_KEY loaded from Colab secrets.")

In [ ]:
# Clone the repository at BRANCH (and EXPECTED_COMMIT, if set), then install exactly
# what requirements.lock pins and the repository without dependencies.
REPO_URL = "https://github.com/magilliam27/MCI-GRU.git"
REPO_DIR = Path("/content/MCI-GRU")


def stream(command, cwd=None, check=True):
    """Run a command, show its output as it arrives, and return its exit code."""
    process = subprocess.Popen(
        command, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    code = process.wait()
    if check and code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {command[:4]}")
    return code


if REPO_DIR.exists():
    raise RuntimeError(
        f"{REPO_DIR} already exists. Use Runtime > Disconnect and delete runtime, "
        "then run the notebook from the top."
    )
stream(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)])
if EXPECTED_COMMIT:
    stream(["git", "-C", str(REPO_DIR), "checkout", "--detach", EXPECTED_COMMIT])
COMMIT = subprocess.run(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
if EXPECTED_COMMIT and COMMIT != EXPECTED_COMMIT:
    raise RuntimeError(f"Checked out {COMMIT}, expected {EXPECTED_COMMIT}")
print("Commit:", COMMIT)
PIP = [sys.executable, "-m", "pip", "install", "-q"]
stream(PIP + ["-r", str(REPO_DIR / "requirements.lock")])
stream(PIP + ["--no-deps", "-e", str(REPO_DIR)])

In [ ]:
# Pull. The PIT file and the LSEG panel are read from Drive and must match the
# committed r1 manifest. Exit status 1 means a blocking finding: the evidence files
# are still written under LOCAL_PACKAGE, but no manifest is published.
lseg = Path(LSEG_PACKAGE_DRIVE_DIR)
exit_code = stream(
    [
        sys.executable, "-m", "scripts.data.export_eodhd_pit_prices",
        "--pit-universe", str(lseg / PIT_UNIVERSE),
        "--reference-panel", str(lseg / REFERENCE_PANEL),
        "--reference-manifest", REFERENCE_MANIFEST,
        "--reference-manifest-sha256", REFERENCE_MANIFEST_SHA256,
        "--start", START,
        "--end", END,
        "--package-root", LOCAL_PACKAGE,
        "--cache-dir", CACHE_DRIVE_DIR,
        "--manifest-output", LOCAL_MANIFEST,
    ],
    cwd=REPO_DIR,
    check=False,
)
print("Pull exit code:", exit_code)

In [ ]:
# Copy the package (and its manifest, when one was published) to Drive. A file that
# already exists there with other bytes stops the copy; nothing is overwritten.
import filecmp
import hashlib
import shutil

target = Path(OUTPUT_DRIVE_DIR)
if exit_code != 0:
    target = target.with_name(target.name + "_FAILED_" + COMMIT[:8])
copied = 0
for source in sorted(Path(LOCAL_PACKAGE).rglob("*")):
    if not source.is_file():
        continue
    destination = target / source.relative_to(LOCAL_PACKAGE)
    if destination.exists():
        if not filecmp.cmp(source, destination, shallow=False):
            raise RuntimeError(f"{destination} exists with other bytes; not overwriting")
        continue
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source, destination)
    copied += 1
manifest = Path(LOCAL_MANIFEST)
if exit_code == 0 and manifest.exists():
    destination = target / manifest.name
    if destination.exists() and not filecmp.cmp(manifest, destination, shallow=False):
        raise RuntimeError(f"{destination} exists with other bytes; not overwriting")
    shutil.copyfile(manifest, destination)
    print("Manifest SHA-256:", hashlib.sha256(manifest.read_bytes()).hexdigest())
print(f"Copied {copied} files to {target}")

## After the pull

- **Exit code 0:** the package and `<PREFIX>_eodhd.r1.json` are on Drive.
  Tell the thread the pull finished. The manifest gets committed to
  `data/manifests/`, and a data config pins its SHA-256, so the first run can
  stage this package the way it stages the LSEG one.
- **Exit code 1:** the files went to a `_FAILED_` folder instead.
  `market/*.meta.json` lists each blocking finding by name, and
  `*_symbols.json` shows every symbol tried. Fix the symbol map
  (`data/mappings/eodhd_symbols_gics_top10_110_2016.json`), then re-run from
  the top; cached responses are reused.